# Feature Analysis — 24 Angles
Compare each feature group across all 24 angles for all 4 mics.
Shows mean ± std per angle to check separability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FEATURES_DIR = r'C:\Users\ahmma\Desktop\farah\features'
ANGLES = list(range(0, 360, 15))

df = pd.read_csv(f'{FEATURES_DIR}/train_DATA.csv')
df['angle'] = df['label'].map({i: a for i, a in enumerate(ANGLES)})

print(f'Rows: {len(df)}  |  Angles: {sorted(df["angle"].unique())}')

## 1. RMS per Mic
Energy level at each mic for each angle. Mic closest to the speaker should have higher RMS.

In [ ]:
mics   = ['rms_mic_right', 'rms_mic_front', 'rms_mic_left', 'rms_mic_back']
labels = ['Right', 'Front', 'Left', 'Back']

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
for ax, col, name in zip(axes, mics, labels):
    g = df.groupby('angle')[col]
    mean, std = g.mean().reindex(ANGLES), g.std().reindex(ANGLES)
    ax.plot(ANGLES, mean, marker='o', linewidth=1.5)
    ax.fill_between(ANGLES, mean - std, mean + std, alpha=0.2)
    ax.set_title(name); ax.set_xlabel('Angle (deg)'); ax.set_ylabel('RMS')
    ax.set_xticks(ANGLES[::2]); ax.tick_params(axis='x', rotation=90)
    ax.grid(True, alpha=0.3)

plt.suptitle('RMS per Mic — 24 Angles', fontsize=13)
plt.tight_layout(); plt.show()

## 2. IPD (Inter-channel Phase Difference)
Phase difference between mic pairs. Should vary smoothly with angle.

In [ ]:
ipd_cols   = ['ipd_pair0', 'ipd_pair1', 'ipd_pair2']
ipd_labels = ['Right-Front', 'Right-Left', 'Front-Back']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, name in zip(axes, ipd_cols, ipd_labels):
    g = df.groupby('angle')[col]
    mean, std = g.mean().reindex(ANGLES), g.std().reindex(ANGLES)
    ax.plot(ANGLES, mean, marker='o', linewidth=1.5, color='orange')
    ax.fill_between(ANGLES, mean - std, mean + std, alpha=0.2, color='orange')
    ax.set_title(name); ax.set_xlabel('Angle (deg)'); ax.set_ylabel('IPD (deg)')
    ax.set_xticks(ANGLES[::2]); ax.tick_params(axis='x', rotation=90)
    ax.grid(True, alpha=0.3)

plt.suptitle('IPD — 24 Angles', fontsize=13)
plt.tight_layout(); plt.show()

## 3. GCC-PHAT TDOA
Time delay between each mic pair. Primary localization feature — each angle should produce a distinct TDOA pattern.

In [ ]:
mic_pairs = ['R-F', 'R-L', 'R-B', 'F-L', 'F-B', 'L-B']
tdoa_cols = [f'gcc_tdoa_{i}' for i in range(6)]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col, name in zip(axes.flatten(), tdoa_cols, mic_pairs):
    g = df.groupby('angle')[col]
    mean, std = g.mean().reindex(ANGLES), g.std().reindex(ANGLES)
    ax.plot(ANGLES, mean, marker='o', linewidth=1.5, color='steelblue')
    ax.fill_between(ANGLES, mean - std, mean + std, alpha=0.2, color='steelblue')
    ax.set_title(f'TDOA {name}'); ax.set_xlabel('Angle (deg)'); ax.set_ylabel('ms')
    ax.set_xticks(ANGLES[::2]); ax.tick_params(axis='x', rotation=90)
    ax.grid(True, alpha=0.3)

plt.suptitle('GCC-PHAT TDOA — 24 Angles', fontsize=13)
plt.tight_layout(); plt.show()

## 4. GCC Strength
Confidence of the TDOA estimate. Higher = cleaner correlation peak.

In [ ]:
str_cols = [f'gcc_strength_{i}' for i in range(6)]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col, name in zip(axes.flatten(), str_cols, mic_pairs):
    g = df.groupby('angle')[col]
    mean, std = g.mean().reindex(ANGLES), g.std().reindex(ANGLES)
    ax.plot(ANGLES, mean, marker='o', linewidth=1.5, color='green')
    ax.fill_between(ANGLES, mean - std, mean + std, alpha=0.2, color='green')
    ax.set_title(f'Strength {name}'); ax.set_xlabel('Angle (deg)'); ax.set_ylabel('Strength')
    ax.set_xticks(ANGLES[::2]); ax.tick_params(axis='x', rotation=90)
    ax.grid(True, alpha=0.3)

plt.suptitle('GCC Strength — 24 Angles', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Silence Detection — Angle 0, All 4 Mics (Raw Audio)
Split the 5min recording into 30ms frames. A frame is silent if its RMS is below a threshold.
Helps verify there are no dead mics or excessive silence before training.

In [ ]:
import wave, os
import numpy as np
import matplotlib.pyplot as plt

DATA      = r'C:\Users\ahmma\Desktop\farah\(24 angles)dataset'
SR        = 16000
FRAME_SEC = 0.03        # 30ms frames
FRAME     = int(SR * FRAME_SEC)
SILENCE_THR = 0.01      # RMS threshold (normalised 0-1)

mics = ['mic_right', 'mic_front', 'mic_left', 'mic_back']

def load_wav(path):
    with wave.open(path) as f:
        d = np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16).astype(np.float32)
    return d / 32768.0

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)

for ax, mic in zip(axes, mics):
    path  = os.path.join(DATA, '0', 'speakerM', '5min', mic + '.wav')
    sig   = load_wav(path)
    t     = np.arange(len(sig)) / SR

    # Frame-level RMS
    n_frames  = len(sig) // FRAME
    frames    = sig[:n_frames * FRAME].reshape(n_frames, FRAME)
    frame_rms = np.sqrt(np.mean(frames ** 2, axis=1))
    is_silent = frame_rms < SILENCE_THR
    silence_pct = is_silent.mean() * 100

    # Plot waveform
    ax.plot(t, sig, linewidth=0.3, color='steelblue', alpha=0.7)

    # Shade silent frames
    for i, silent in enumerate(is_silent):
        if silent:
            ax.axvspan(i * FRAME_SEC, (i + 1) * FRAME_SEC,
                       color='red', alpha=0.3, linewidth=0)

    ax.axhline(SILENCE_THR,  color='red', linestyle='--', linewidth=0.8, label=f'thr={SILENCE_THR}')
    ax.axhline(-SILENCE_THR, color='red', linestyle='--', linewidth=0.8)
    ax.set_ylabel(mic)
    ax.set_title(f'{mic}  —  silence: {silence_pct:.1f}%', fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Silence Detection — Angle 0, 4 Mics (red = silent frame)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Silence Detection per Angle (from CSV)
A chunk is silent if the mean RMS across all 4 mics is below the threshold.
Shows each angle separately — helps spot dead mic or noisy angle issues.

In [ ]:
SILENCE_THR = 50    # raw int16 RMS — adjust if needed
RMS_COLS    = ['rms_mic_right', 'rms_mic_front', 'rms_mic_left', 'rms_mic_back']
MIC_COLORS  = ['steelblue', 'orange', 'green', 'red']
ANGLES      = list(range(0, 360, 15))

df['mean_rms'] = df[RMS_COLS].mean(axis=1)
df['silent']   = df['mean_rms'] < SILENCE_THR
df['angle']    = df['label'].map({i: a for i, a in enumerate(ANGLES)})

fig, axes = plt.subplots(6, 4, figsize=(20, 18), sharey=False)

for ax, angle in zip(axes.flatten(), ANGLES):
    sub = df[df['angle'] == angle].reset_index(drop=True)

    for col, color, mic in zip(RMS_COLS, MIC_COLORS, ['Right','Front','Left','Back']):
        ax.plot(sub.index, sub[col], linewidth=0.8, color=color, alpha=0.7, label=mic)

    # Shade silent chunks
    for idx, row in sub[sub['silent']].iterrows():
        ax.axvspan(idx - 0.5, idx + 0.5, color='gray', alpha=0.4, linewidth=0)

    ax.axhline(SILENCE_THR, color='black', linestyle='--', linewidth=0.8)
    n_silent = sub['silent'].sum()
    ax.set_title(f'{angle}deg  silent:{n_silent}/{len(sub)}', fontsize=8)
    ax.set_xlabel('Chunk', fontsize=7); ax.set_ylabel('RMS', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2)

axes[0, 0].legend(fontsize=6, loc='upper right')
plt.suptitle(f'Silence Detection — All 24 Angles (thr={SILENCE_THR})', fontsize=13)
plt.tight_layout()
plt.show()

total_silent = df['silent'].sum()
print(f'Total silent chunks: {total_silent}/{len(df)}  ({total_silent/len(df)*100:.1f}%)')
print(df.groupby('angle')['silent'].sum().rename('silent_chunks'))